In [1]:
%%writefile cryptovault.py
import argparse
import os
import sys

ENGLISH_FREQS = {
    'A': 0.0817, 'B': 0.0149, 'C': 0.0278, 'D': 0.0425, 'E': 0.1270, 'F': 0.0223,
    'G': 0.0202, 'H': 0.0609, 'I': 0.0697, 'J': 0.0015, 'K': 0.0077, 'L': 0.0403,
    'M': 0.0241, 'N': 0.0675, 'O': 0.0751, 'P': 0.0193, 'Q': 0.0010, 'R': 0.0599,
    'S': 0.0633, 'T': 0.0906, 'U': 0.0276, 'V': 0.0098, 'W': 0.0236, 'X': 0.0015,
    'Y': 0.0197, 'Z': 0.0007
}

def caesar_transform(text: str, shift: int) -> str:
    result = []
    for char in text:
        if char.isalpha():
            start = ord('A') if char.isupper() else ord('a')
            shifted_char = chr(start + (ord(char) - start + shift) % 26)
            result.append(shifted_char)
        else:
            result.append(char)
    return "".join(result)

def calculate_score(text: str) -> float:
    letters = [char.upper() for char in text if char.isalpha()]
    if not letters:
        return float('inf')
    counts = {ch: 0 for ch in ENGLISH_FREQS}
    for ch in letters:
        counts[ch] += 1
    total_letters = len(letters)
    chi_squared = 0.0
    for ch, expected_prop in ENGLISH_FREQS.items():
        expected_count = expected_prop * total_letters
        observed_count = counts[ch]
        chi_squared += ((observed_count - expected_count) ** 2) / expected_count
    return chi_squared

def crack_ciphertext(ciphertext: str):
    candidates = []
    for shift in range(26):
        decrypted_candidate = caesar_transform(ciphertext, -shift)
        score = calculate_score(decrypted_candidate)
        candidates.append((score, shift, decrypted_candidate))
    candidates.sort(key=lambda x: x)
    print("\n--- TOP 3 LIKELY PLAINTEXTS ---")
    for i in range(min(3, len(candidates))):
        score, shift, text = candidates[i]
        print(f"Rank {i+1} (Key/Shift used to decrypt: {shift}):")
        print(f"Text: \"{text}\"\n")

def main():
    parser = argparse.ArgumentParser(description="CryptoVault CLI Tool")
    parser.add_argument('mode', choices=['encrypt', 'decrypt', 'crack'])
    parser.add_argument('file_or_text')
    parser.add_argument('--shift', type=int, default=0)
    args = parser.parse_args()

    if args.mode == 'crack':
        if os.path.exists(args.file_or_text):
            with open(args.file_or_text, 'r', encoding='utf-8') as f:
                content = f.read()
        else:
            content = args.file_or_text
        crack_ciphertext(content)
        return

    if not os.path.exists(args.file_or_text):
        print(f"Error: File '{args.file_or_text}' not found.", file=sys.stderr)
        sys.exit(1)

    with open(args.file_or_text, 'r', encoding='utf-8') as f:
        file_data = f.read()

    if args.mode == 'encrypt':
        print(caesar_transform(file_data, args.shift))
    elif args.mode == 'decrypt':
        print(caesar_transform(file_data, -args.shift))

if __name__ == "__main__":
    main()


Writing cryptovault.py
